# Klasifikasi Bunga: CNN dari Nol vs Transfer Learning (MobileNetV2)

**Tugas Individu Deep Learning**

- Nama: *(isi nama)*
- NIM: *(isi NIM)*
- Dataset: TF Flowers, 5 kelas (daisy, dandelion, roses, sunflowers, tulips), 3.670 gambar, split 80% train / 20% validasi
- Model: (1) CNN custom, (2) Transfer Learning MobileNetV2

**Tujuan:** membandingkan CNN yang dilatih dari nol dengan transfer learning pada dataset gambar yang sama, dari sisi akurasi, waktu training, dan overfitting.

## 1. Import Library

In [ ]:
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Download dan Load Dataset

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archive = tf.keras.utils.get_file("flower_photos.tar", origin=URL, extract=True)

# Cari folder yang langsung berisi subfolder kelas (daisy, dandelion, dst.)
data_dir = None
for root, dirs, _ in os.walk(os.path.dirname(archive)):
    if {"daisy", "roses", "tulips"}.issubset(set(dirs)):
        data_dir = root
        break
assert data_dir is not None, "Folder kelas bunga tidak ditemukan"
print("Folder data:", data_dir, "| isi:", sorted(os.listdir(data_dir)))

IMG_SIZE = (160, 160)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH)
val_raw = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
assert NUM_CLASSES == 5, f"Harusnya 5 kelas, terbaca {NUM_CLASSES}: {class_names}"
print("Kelas:", class_names)

# Materialisasi data validasi sekali saja agar urutannya tetap
# (penting supaya label asli cocok dengan prediksi saat evaluasi).
val_images = np.concatenate([x.numpy() for x, _ in val_raw]).astype("float32")
y_true = np.concatenate([y.numpy() for _, y in val_raw]).astype(int)
val_ds = tf.data.Dataset.from_tensor_slices((val_images, y_true)).batch(BATCH)
print("Train batch:", len(train_ds), "| Val gambar:", len(y_true))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.suptitle("Contoh gambar dataset")
plt.tight_layout()
plt.show()

## 3. Model 1: CNN Custom (dari nol)

Arsitektur: data augmentation, rescaling, 3 blok `Conv2D + MaxPooling`, `Flatten`, `Dropout`, lalu `Dense` softmax (5 kelas).

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

def build_cnn():
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = layers.Rescaling(1./255)(x)
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="cnn_custom")

cnn = build_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

In [ ]:
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)

t0 = time.time()
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[early])
time_cnn = time.time() - t0
print(f"Waktu training CNN: {time_cnn:.1f} detik")

## 4. Model 2: Transfer Learning (MobileNetV2)

Tahap 1 (*feature extraction*): base model MobileNetV2 pretrained ImageNet dibekukan, hanya head baru yang dilatih.
Tahap 2 (*fine-tuning*): 30 layer terakhir base model dibuka dan dilatih dengan learning rate kecil.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False

def build_tl():
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = layers.Rescaling(1./127.5, offset=-1)(x)   # skala [-1, 1] sesuai MobileNetV2
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="mobilenetv2_tl")

tl = build_tl()
tl.compile(optimizer=keras.optimizers.Adam(1e-3),
           loss="sparse_categorical_crossentropy", metrics=["accuracy"])
tl.summary()

In [ ]:
# Tahap 1: feature extraction
t0 = time.time()
hist_tl1 = tl.fit(train_ds, validation_data=val_ds, epochs=8)
time_tl1 = time.time() - t0
print(f"Waktu tahap 1: {time_tl1:.1f} detik")

In [ ]:
# Tahap 2: fine-tuning 30 layer terakhir
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

tl.compile(optimizer=keras.optimizers.Adam(1e-5),
           loss="sparse_categorical_crossentropy", metrics=["accuracy"])

t0 = time.time()
hist_tl2 = tl.fit(train_ds, validation_data=val_ds, epochs=5)
time_tl2 = time.time() - t0
time_tl = time_tl1 + time_tl2
print(f"Waktu fine-tuning: {time_tl2:.1f} detik | total TL: {time_tl:.1f} detik")

## 5. Kurva Training

In [ ]:
def merge(*hists):
    out = {}
    for k in ("accuracy", "val_accuracy", "loss", "val_loss"):
        out[k] = sum([h.history[k] for h in hists], [])
    return out

h_cnn = merge(hist_cnn)
h_tl = merge(hist_tl1, hist_tl2)
ft_start = len(hist_tl1.history["accuracy"])

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for row, (name, h) in enumerate([("CNN Custom", h_cnn), ("MobileNetV2 (TL)", h_tl)]):
    ax[row, 0].plot(h["accuracy"], label="train"); ax[row, 0].plot(h["val_accuracy"], label="val")
    ax[row, 0].set_title(f"{name}: Accuracy"); ax[row, 0].legend()
    ax[row, 1].plot(h["loss"], label="train"); ax[row, 1].plot(h["val_loss"], label="val")
    ax[row, 1].set_title(f"{name}: Loss"); ax[row, 1].legend()
    if row == 1:
        for a in ax[row]: a.axvline(ft_start - 1, color="gray", ls="--", label="mulai fine-tuning")
for a in ax.flat: a.set_xlabel("Epoch")
plt.tight_layout()
os.makedirs("images", exist_ok=True)
plt.savefig("images/training_curves.png", dpi=150)
plt.show()

## 6. Evaluasi: Confusion Matrix dan Classification Report

In [ ]:
def evaluate(model, name):
    probs = model.predict(val_ds, verbose=0)
    y_pred = probs.argmax(axis=1)
    print(f"=== {name} ===")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    cm = confusion_matrix(y_true, y_pred)
    return y_pred, cm

pred_cnn, cm_cnn = evaluate(cnn, "CNN Custom")
pred_tl, cm_tl = evaluate(tl, "MobileNetV2 (Transfer Learning)")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, cm, t in zip(ax, [cm_cnn, cm_tl], ["CNN Custom", "MobileNetV2 (TL)"]):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names,
                yticklabels=class_names, ax=a)
    a.tick_params(axis="x", rotation=45)
    a.set_title(t); a.set_xlabel("Prediksi"); a.set_ylabel("Aktual")
plt.tight_layout()
plt.savefig("images/confusion_matrix.png", dpi=150)
plt.show()

## 7. Contoh Prediksi Salah (Error Analysis)

In [ ]:
images_all = val_images.astype("uint8")
wrong = np.where(pred_tl != y_true)[0][:8]

plt.figure(figsize=(12, 6))
for i, idx in enumerate(wrong):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images_all[idx])
    plt.title(f"Aktual: {class_names[y_true[idx]]}\nPred: {class_names[pred_tl[idx]]}", fontsize=9)
    plt.axis("off")
plt.suptitle("Kesalahan prediksi MobileNetV2")
plt.tight_layout()
plt.savefig("images/wrong_predictions.png", dpi=150)
plt.show()

## 8. Perbandingan Model

In [ ]:
import pandas as pd

def summary_row(name, model, h, t):
    return {
        "Model": name,
        "Parameter": f"{model.count_params():,}",
        "Train Acc": round(h["accuracy"][-1], 4),
        "Val Acc": round(h["val_accuracy"][-1], 4),
        "Gap (Train-Val)": round(h["accuracy"][-1] - h["val_accuracy"][-1], 4),
        "Val Loss": round(h["val_loss"][-1], 4),
        "Waktu Training (s)": round(t, 1),
    }

df = pd.DataFrame([
    summary_row("CNN Custom", cnn, h_cnn, time_cnn),
    summary_row("MobileNetV2 (TL)", tl, h_tl, time_tl),
])
df

## 9. Analisis dan Kesimpulan

> **Isi bagian ini dengan angka dari hasil run kamu sendiri** (tabel di atas).

1. **Akurasi:** Model mana yang lebih tinggi pada data validasi, dan selisihnya berapa?
2. **Overfitting:** Lihat gap train vs val dan kurva loss. Apakah CNN custom lebih overfit? Apakah augmentasi dan dropout membantu?
3. **Waktu dan ukuran model:** Bandingkan waktu training dan jumlah parameter. Berapa banyak parameter MobileNetV2 yang benar-benar dilatih?
4. **Error analysis:** Gambar seperti apa yang sering salah diprediksi (objek kecil, latar ramai, pose tidak biasa)?
5. **Kesimpulan:** Transfer learning umumnya lebih unggul pada dataset kecil karena fitur visual dari ImageNet sudah dipelajari sebelumnya, sedangkan CNN dari nol harus mempelajari semuanya dari 2.936 gambar. Konfirmasi atau bantah hal ini dengan hasil eksperimenmu.